# Data Preparation

- Feature Selection
- Feature Transfromation
- Handling Missing Values
- Handling Outliers (after outlier detection)

In [18]:
import pandas as pd
import numpy as np

In [19]:
df = pd.read_csv('..//dataset/cmi_internet.csv')

## 1. Feature Selection

In [20]:
# Removing Season Columns from the Dataset

season_columns = [col for col in df.columns if 'Season' in col]

new_df = df.drop(columns=season_columns)

# Removing PCIAT columns

pci_columns = [col for col in df.columns if 'PCIAT' in col and 'Season' not in col]

new_df = new_df.drop(columns=pci_columns)

# Removing Physical-BMI (= computed BMI + random noise, it is redundant)

new_df = new_df.drop(columns=['Physical-BMI'])

# Removing BIA-BIA_BMI (= it doesn't corelate with the computed BMI, it's redundant)

new_df = new_df.drop(columns=['BIA-BIA_BMI'])


Handling duplicate PAQ_A-PAQ_A_Total/PAQ_C-PAQ_C_Total

In [21]:
# select records that have values both in PAQ_A-PAQ_A_Total and PAQ_C-PAQ_C_Total
filtered_df = new_df.dropna(subset=['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total'])
# choose one of the two columns to keep: 
# 1) keep PAQ_A-PAQ_A_Total and drop PAQ_C-PAQ_C_Total if age > 12
# 2) keep PAQ_C-PAQ_C_Total and drop PAQ_A-PAQ_A_Total if age <= 12
filtered_df.loc[filtered_df['Basic_Demos-Age'] > 12, 'PAQ_C-PAQ_C_Total'] = np.nan
filtered_df.loc[filtered_df['Basic_Demos-Age'] <= 12, 'PAQ_A-PAQ_A_Total'] = np.nan
# drop the original columns
filtered_df = filtered_df.drop(columns=['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total'])
# rename the new columns to a common name, e.g., 'PAQ_Total'
filtered_df = filtered_df.rename(columns={'PAQ_A-PAQ_A_Total': 'PAQ_Total', 'PAQ_C-PAQ_C_Total': 'PAQ_Total'})
# update the original dataframe with the filtered dataframe
new_df.update(filtered_df)



## 3. Feature Transformation

In [22]:
# Transforming Fitness_Endurance-Time_Mins in seconds and creating a new column called Fitness_Endurance-Time containing  the sum of Fitness_Endurance-Time_Mins and Fitness_Endurance-Time_Secs

# fill missing values of seconds with 0
new_df['Fitness_Endurance-Time_Sec'] = new_df['Fitness_Endurance-Time_Sec'].fillna(0)
new_df['Fitness_Endurance-Time'] = new_df['Fitness_Endurance-Time_Mins'] * 60 + new_df['Fitness_Endurance-Time_Sec']

new_df = new_df.drop(columns=['Fitness_Endurance-Time_Mins', 'Fitness_Endurance-Time_Sec'])
print(len(new_df[new_df['Fitness_Endurance-Time'].isnull()]))

2982


In [23]:
new_df.shape

(8460, 47)

In [24]:
# number of rows wothout missing values
df_no_missing = new_df.dropna()
print("Number of rows without missing values:", len(df_no_missing))

Number of rows without missing values: 82


In [25]:
print("Correlation between BIA-BIA_FMI and BIA-BIA_Fat:", df_no_missing['BIA-BIA_FMI'].corr(df_no_missing['BIA-BIA_Fat']))
print("Correlation between BIA-BIA_FMI and Physical-Height:", df_no_missing['BIA-BIA_FMI'].corr(df_no_missing['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and Physical-Height:", df_no_missing['BIA-BIA_FFMI'].corr(df_no_missing['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and BIA-BIA_FFM:", df_no_missing['BIA-BIA_FFMI'].corr(df_no_missing['BIA-BIA_FFM']))

Correlation between BIA-BIA_FMI and BIA-BIA_Fat: 0.9751604070175767
Correlation between BIA-BIA_FMI and Physical-Height: 0.32589743428997414
Correlation between BIA-BIA_FFMI and Physical-Height: 0.5722177425719592
Correlation between BIA-BIA_FFMI and BIA-BIA_FFM: 0.8214244444341205


## 2. Missing Value Correction Strategy

2.1 Semantic Strategy

Filling missing values through theoretical formulae:

- BIA-BIA_TBW = BIA-BIA_ICW + BIA-BIA_ECW = 0.73 * FFM
- BIA-BIA_DEE = BIA-BIA_BMR * BIA-BIA_Activity_Level_num
- BIA-BIA_FFM = Physical-Weight - BIA-BIA_Fat = BIA-BIA_FFMI * Height (m)^2 = BIA-BIA_LST + BIA-BIA_BMC
- BIA-BIA_Fat = BIA-BIA_FMI * Height (m)^2
- BIA-BIA_FMI = BIA-BIA_Fat (kg)/Height (m)^2 = BMI - BIA-BIA_FFMI
- BIA-BIA_FFMI = BIA-BIA_FFM/Height (m)^2 = BMI - BIA-BIA_FM

In [26]:
# check whether BIA-BIA_FFM and BIA-BIA_FMI are correlated with Physical-Weight and Physical-Height
# print("Correlation between BIA-BIA_FFM and Physical-Weight:", new_df['BIA-BIA_FFM'].corr(new_df['Physical-Weight']))
# print("Correlation between BIA-BIA_FFM and Physical-Height:", new_df['BIA-BIA_FFM'].corr(new_df['Physical-Height']))
print("Correlation between BIA-BIA_FMI and BIA-BIA_Fat:", new_df['BIA-BIA_FMI'].corr(new_df['BIA-BIA_Fat']))
print("Correlation between BIA-BIA_FMI and Physical-Height:", new_df['BIA-BIA_FMI'].corr(new_df['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and Physical-Height:", new_df['BIA-BIA_FFMI'].corr(new_df['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and BIA-BIA_FFM:", new_df['BIA-BIA_FFMI'].corr(new_df['BIA-BIA_FFM']))

Correlation between BIA-BIA_FMI and BIA-BIA_Fat: 0.11127898817596941
Correlation between BIA-BIA_FMI and Physical-Height: 0.28680140322975195
Correlation between BIA-BIA_FFMI and Physical-Height: 0.1807378311885216
Correlation between BIA-BIA_FFMI and BIA-BIA_FFM: 0.04822768443524458


In [27]:
def impute_bia_values(df):
    # 1. Preparazione costanti di supporto (non sovrascriviamo le originali)
    # Convertiamo altezza in metri e peso in kg per i calcoli
    h_m = df['Physical-Height'] * 0.0254 # from inches to meters
    w_kg = df['Physical-Weight'] * 0.453592 # from pounds to kg

    # --- REGOLA 1: TBW (Total Body Water) ---
    # Se manca TBW, usa la somma di ICW ed ECW
    mask_tbw = df['BIA-BIA_TBW'].isna()
    df.loc[mask_tbw, 'BIA-BIA_TBW'] = df['BIA-BIA_ICW'] + df['BIA-BIA_ECW']

    mask_icw = df['BIA-BIA_ICW'].isna()
    df.loc[mask_icw, 'BIA-BIA_ICW'] = df['BIA-BIA_TBW'] - df['BIA-BIA_ECW']

    mask_ecw = df['BIA-BIA_ECW'].isna()
    df.loc[mask_ecw, 'BIA-BIA_ECW'] = df['BIA-BIA_TBW'] - df['BIA-BIA_ICW']

    # --- REGOLA 2: FFM (Fat Free Mass) ---
    # Opzione A: Weight - Fat Mass (se entrambi sono presenti)  
    mask_ffm = df['BIA-BIA_FFM'].isna()
    df.loc[mask_ffm, 'BIA-BIA_FFM'] = df['Physical-Weight'] - df['BIA-BIA_Fat']
    
    # Opzione B: Se manca ancora, usa l'indice FFMI (se presente)
    mask_ffm_still_na = df['BIA-BIA_FFM'].isna()
    df.loc[mask_ffm_still_na, 'BIA-BIA_FFM'] = df['BIA-BIA_FFMI'] * (h_m**2)

    # --- REGOLA 3: Fat Percentage (BIA-BIA_Fat) ---
    # Nel dataset CMI il campo è 'BIA-BIA_Fat' (MASS part)
    mask_fat = df['BIA-BIA_Fat'].isna()
    
    # Calcoliamo prima la massa grassa (kg) in modo temporaneo

    # se il valore di FFM è presente, usiamo la formula: Fat Mass = Weight - FFM
    df.loc[mask_fat, 'BIA-BIA_Fat'] =  df['Physical-Weight'] - df['BIA-BIA_FFM']
    # Altrimenti, se non abbiamo FFM, proviamo a usare l'indice FMI
    if df['BIA-BIA_Fat'].isna().any():
        # select only rows where BIA-BIA_Fat is still NA
        mask_fat_still_na = df['BIA-BIA_Fat'].isna()
        df.loc[mask_fat_still_na, 'BIA-BIA_Fat'] = df['BIA-BIA_FMI'] * (h_m**2)

    # --- REGOLA 4: DEE (Daily Energy Expenditure) ---
    # Nota: Usiamo i moltiplicatori standard della letteratura medica (Equazione di Harris-Benedict)
    # 1=1.2 (Sedentario), 2=1.375, 3=1.55, 4=1.725, 5=1.9 (Atleta)

    activity_map = {1: 1.2, 2: 1.375, 3: 1.55, 4: 1.725, 5: 1.9}
    multipliers = df['BIA-BIA_Activity_Level_num'].map(activity_map)
    
    mask_dee = df['BIA-BIA_DEE'].isna()
    df.loc[mask_dee, 'BIA-BIA_DEE'] = df['BIA-BIA_BMR'] * multipliers

    return df

2.2 Redundant Feature Elimination

In [28]:
# removing redundant features: Physical-BMI, all the "Zone" columns, BIA-BIA_BMI, BIA-BIA_TBW, BIA-BIA_FMI, BIA-BIA_FFMI
new_df = new_df.drop(columns=[ "id", 'BIA-BIA_TBW', 'BIA-BIA_FMI', 'BIA-BIA_FFMI', "SDS-SDS_Total_Raw", "BIA-BIA_DEE", "BIA-BIA_FFM", "Fitness_Endurance-Max_Stage","sii"] + [col for col in new_df.columns if 'Zone' in col])

In [29]:
new_df.shape

(8460, 31)

In [30]:
new_df.columns

Index(['Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference',
       'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP',
       'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU',
       'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR',
       'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW',
       'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'PAQ_A-PAQ_A_Total',
       'PAQ_C-PAQ_C_Total', 'SDS-SDS_Total_T',
       'PreInt_EduHx-computerinternet_hoursday', 'Fitness_Endurance-Time'],
      dtype='object')

2.3 MICE

In [32]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# 1. Selected features for imputation (those with missing values and relevant for analysis)
features = [
    'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score', 
    'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference', 
    'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP', 
    'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU', 
    'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL', 
    'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR', 
    'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW', 
    'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'PAQ_A-PAQ_A_Total', 
    'PAQ_C-PAQ_C_Total', 'SDS-SDS_Total_T', 
    'PreInt_EduHx-computerinternet_hoursday', 'Fitness_Endurance-Time'
]


X = new_df[features].copy()

# 2. Iterative Imputer Configuration
# Let's use RandomForestRegressor to handle the complexity of biological data and capture non-linear relationships
imputer_rf = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=50, # Reducing the number of trees for faster imputation and to avoid overfitting, can be increased if needed
        max_depth=10,     # Limiting depth to prevent overfitting on small datasets
        min_samples_leaf=5, # Adding min_samples_leaf to prevent overfitting on small datasets
        n_jobs=-1,
        random_state=42
    ),
    max_iter=15,
    tol=0.01,            # Higher tolerance to speed up convergence, can be reduced for more precision
    random_state=42,
    initial_strategy='median'
)

# 3. Execution
print("Beginning imputation (this may take a few minutes)...")
X_imputed_array = imputer_rf.fit_transform(X)

# 4. Recreate the clean DataFrame
df_imputed = pd.DataFrame(X_imputed_array, columns=features)


Beginning imputation (this may take a few minutes)...


c:\Users\leona\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [33]:
print("Statistiche pre-imputazione:\n", X['Physical-Weight'].describe())
print("\nStatistiche post-imputazione:\n", df_imputed['Physical-Weight'].describe())

Statistiche pre-imputazione:
 count    7641.000000
mean       84.428737
std        40.218441
min         0.000000
25%        55.200000
50%        75.000000
75%       107.000000
max       315.000000
Name: Physical-Weight, dtype: float64

Statistiche post-imputazione:
 count    8460.000000
mean       83.748794
std        39.184002
min         0.000000
25%        55.800000
50%        75.800000
75%       105.400000
max       315.000000
Name: Physical-Weight, dtype: float64


In [16]:
# save the imputed dataset to a new CSV file
df_imputed.to_csv('..//dataset/cmi_internet_imputed.csv', index=False)

In [ ]:

# --- 5. Recompute Deterministic Columns ---

# Metric conversions
h_m = df_imputed['Physical-Height'] * 0.0254
w_kg = df_imputed['Physical-Weight'] * 0.453592

# BMI = Weight (kg) / (Height (m))^2
df_imputed['Physical-BMI'] = w_kg / (h_m ** 2)

# BIA: Total Body Water (TBW = ICW + ECW)
df_imputed['BIA-BIA_TBW'] = df_imputed['BIA-BIA_ICW'] + df_imputed['BIA-BIA_ECW']

# BIA: Fat Free Mass (FFM = LST + BMC)
df_imputed['BIA-BIA_FFM'] = df_imputed['BIA-BIA_LST'] + df_imputed['BIA-BIA_BMC']

# BIA: Daily Energy Expenditure
activity_map = {1: 1.2, 2: 1.375, 3: 1.55, 4: 1.725, 5: 1.9}
df_imputed['BIA-BIA_DEE'] = df_imputed['BIA-BIA_BMR'] * df_imputed['BIA-BIA_Activity_Level_num'].map(activity_map)

# BIA: Fat Mass Index (FMI)
# Fat_Mass_kg = Weight (kg) * (Fat_% / 100)
fat_mass_kg = w_kg * (df_imputed['BIA-BIA_Fat'] / 100)
df_imputed['BIA-BIA_FMI'] = fat_mass_kg / (h_m ** 2)

# SDS: Raw Score 
# Let's keep only the T-score for analysis, as the raw score is often not used directly in modeling.

print("Imputazione e ricalcolo completati con successo.")